In [11]:
# ==========================================
#   IMPORT LIBRARIES & SETUP
# ==========================================
%load_ext autoreload
%autoreload 2

from metrics import evaluate
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # implicit recommends this for ALS

from implicit.als import AlternatingLeastSquares

import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import implicit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## DATA LOADING

In [2]:
articles=pd.read_parquet('data/articles_processed.parquet')
customers=pd.read_parquet('data/customers_processed.parquet')
transactions=pd.read_parquet('data/transactions_processed.parquet')


In [5]:
#=================================================================================
#                        MAPPING
#=================================================================================

unique_customers = customers['customer_id'].unique()
unique_articles = articles['article_id'].unique()

# Mappings
customer2idx = {c: i for i, c in enumerate(unique_customers)}
article2idx = {a: i for i, a in enumerate(unique_articles)}

# Reverse Mappings 
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {i: a for a, i in article2idx.items()}

# Mapping transactions-ի վրա
transactions['user_idx'] = transactions['customer_id'].map(customer2idx)
transactions['item_idx'] = transactions['article_id'].map(article2idx)

# Check: no unmapped IDs
print(transactions['user_idx'].isna().sum(), transactions['item_idx'].isna().sum())

0 0


In [6]:
#================================================================================
#                  TRAIN-VALIDATION SPLIT
#================================================================================
last_date = transactions['t_dat'].max()
val_start = last_date - pd.Timedelta(days=6)
train = transactions[transactions['t_dat'] < val_start]
val = transactions[transactions['t_dat'] >= val_start]

print(transactions['t_dat'].dtype)
print(len(train), len(val))
val_customers = set(val['user_idx'].unique())
train_customers = set(train['user_idx'].unique())
print(len(val_customers), len(val_customers - train_customers))

datetime64[ns]
31548013 240311
68984 5572


* val-ում 68 984  հաճախորդ կա, որոնցից 5 572-ը train-ում չկան

## Collaborative Filtering

In [7]:
#=================================================================================
#                              CSR Matrix
#=================================================================================

train_user_item = train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')

rows = train_user_item['user_idx'].values
cols = train_user_item['item_idx'].values
weights = train_user_item['weight'].values.astype(np.float32)

n_users = len(customer2idx)
n_items = len(article2idx)


user_item_matrix = csr_matrix((weights, (rows, cols)), shape=(n_users, n_items))


In [8]:

recent_popular = train[train['t_dat'] >= train['t_dat'].max() - pd.Timedelta(days=6)]
fallback_top12 = recent_popular.groupby('article_id').size().nlargest(12).index.tolist()

def predict_implicit_model(model, val_df, k=12):
    
    actuals_dict = val_df.groupby('customer_id')['article_id'].unique().apply(set).to_dict()
    val_customers = list(actuals_dict.keys())

    user_idxs = np.array([customer2idx[c_id] for c_id in val_customers])

    has_history = np.asarray(user_item_matrix[user_idxs].getnnz(axis=1)) > 0

    ids, scores = model.recommend(
        user_idxs,
        user_item_matrix[user_idxs],
        N=k,
        filter_already_liked_items=False
    )

    predictions = {}
    for i, c_id in enumerate(val_customers):
        if has_history[i]:
            recs = [idx2article[int(idx)] for idx in ids[i] if idx >= 0]
        else:
            recs = []  # cold customer
        fill = [p for p in fallback_top12 if p not in recs]
        predictions[c_id] = (recs + fill)[:k]

    return predictions, actuals_dict

In [9]:
#=================================================================================
#                   Item-Item Cosine Similarity
#=================================================================================

cosine_model = implicit.nearest_neighbours.CosineRecommender(K=20)
cosine_model.fit(user_item_matrix)
print("Cosine Similarity model successfully fitted!")


C:\Users\user\AppData\Roaming\Python\Python314\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.21536946296691895 seconds
  warnings.warn(
100%|██████████| 105542/105542 [00:06<00:00, 16051.45it/s]


Cosine Similarity model successfully fitted!


In [10]:
cosine_preds, actuals_dict = predict_implicit_model(cosine_model, val, k=12)

total_catalog_items = articles['article_id'].nunique()
sales_series = train['article_id'].value_counts()

total_sales = len(train)

novelty_dict = (
    (-np.log2(sales_series / total_sales)).to_dict()
)

cosine_results = evaluate(cosine_preds, actuals_dict, total_catalog_items,novelty_dict, k=12)

print("--- Cosine Similarity Results ---")
for metric_name, value in cosine_results.items():
    print(f"{metric_name}: {value:.5f}")

--- Cosine Similarity Results ---
Precision@12: 0.00490
Recall@12: 0.02707
MAP@12: 0.01092
NDCG@12: 0.02638
Novelty@12: 14.76695
Coverage: 0.68818


In [12]:
#=================================================================================
#                              ALS
#=================================================================================

als_matrix = user_item_matrix.copy()
als_matrix.data = 1 + np.log(als_matrix.data)

als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=15,
    alpha=10,
    random_state=42
)
als_model.fit(als_matrix)

print(als_model.user_factors.shape, als_model.item_factors.shape)   


als_preds, actuals_dict = predict_implicit_model(als_model, val, k=12)
als_results = evaluate(als_preds, actuals_dict, total_catalog_items, novelty_dict, k=12)

for metric_name, value in als_results.items():
    print(f"{metric_name}: {value:.5f}")

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [01:11<00:00,  4.79s/it]


(1371980, 64) (105542, 64)
Precision@12: 0.00447
Recall@12: 0.02061
MAP@12: 0.00814
NDCG@12: 0.02450
Novelty@12: 12.11340
Coverage: 0.02559
